In [19]:
def correct_date(row):
    if not isinstance(row["FECHA"], str):
        return row["FECHA"]

    if row["FECHA"][0] == "'":
        row["FECHA"] = row["FECHA"][1:]
    if "/" in row["FECHA"]:
        date_split = row["FECHA"].split("/")
    if "-" in row["FECHA"]:
        date_split = row["FECHA"].split("-")
    try:
        day = date_split[0].zfill(2)
        month = date_split[1].zfill(2)
        all_date = day + "/" + month + "/" + date_split[2]
        return all_date
    except Exception as e:
        raise ValueError(f"Error processing date: {row['FECHA']}. Error: {e}")

def correct_bill_number(row):
    if not isinstance(row["FACTURA"], str):
        return row["FACTURA"]
    row["FACTURA"] = row["FACTURA"].strip()
    row["FACTURA"] = row["FACTURA"].replace(" ", "")
    return row["FACTURA"]

def correct_numeric(row, column):
    if not isinstance(row[column], str):
        return row[column]
    row[column] = row[column].replace("$", "").replace(",", ".").replace(" ", "")
    return row[column]

In [21]:
import pandas as pd
import os

main = os.getcwd().replace("\\", "/").split("notebooks")[0]

all_df = pd.read_excel(main + "data/bills_2025_REVISADAS_II.xlsx")
all_df["FACTURA"] = all_df.apply(lambda row: correct_bill_number(row)  , axis=1)

duplicados = all_df[all_df.duplicated("FACTURA", keep=False)]
no_duplicados = all_df[~all_df.duplicated("FACTURA", keep=False)]

dup_ord = duplicados.sort_values(by=["EXTRACTION"], ascending=False)

dup_ord = dup_ord.dropna(subset=["FACTURA"])

res = dup_ord.drop_duplicates(subset=["FACTURA"], keep="first")

final = pd.concat([res, no_duplicados], ignore_index=True)

final = final[final["CHECK"] != "Sin nombre"]

final = final[["FECHA", "PROVEEDOR", "NIT", "FACTURA","VALOR ANTES DE IVA", "IVA", "TOTAL", "CHECK"]]
final = final.dropna()
final["FECHA"] = final.apply(lambda row: correct_date(row)  , axis=1)
final["FECHA"] = pd.to_datetime(final["FECHA"], format="mixed", dayfirst=True).dt.strftime("%d/%m/%Y")
num_columns = ["VALOR ANTES DE IVA", "IVA", "TOTAL"]
for col in num_columns:
    final[col] = final.apply(lambda row: correct_numeric(row, col)  , axis=1)
    final[col] = pd.to_numeric(final[col], errors='coerce')

In [22]:
final.to_excel(main + "data/iva_armando.xlsx", index=False)

In [12]:
import pandas as pd
import os

main = os.getcwd().replace("\\", "/").split("notebooks")[0]
all_df = pd.read_excel(main + "data/bills_2025_REVISADAS.xlsx")

In [4]:
from tqdm.auto import tqdm
from src.scripts.utils.ask_a_model import query_model_structure_xml

error_404 = all_df[all_df["CHECK"] == 404]

file_names = error_404["FILE_NAME"].unique().tolist()

extracted_path = os.path.join(main, "data", "extracted")
invoices = []

for file in tqdm(file_names, desc="Processing files", unit="file", total=len(file_names)):
    total_path = os.path.join(extracted_path, file + ".xml")
    if not os.path.exists(total_path):
        continue
    with open(total_path, "r", encoding="utf-8") as f:
        content = f.read()

    invoice_data = query_model_structure_xml(content)
    invoices.append((file, invoice_data))

Processing files:   0%|          | 0/29 [00:00<?, ?file/s]

In [9]:
for file, data in invoices:

    data = data.model_dump()
    all_df.loc[all_df["FILE_NAME"] == file, "FACTURA"] = data.get("numero_factura", None)
    all_df.loc[all_df["FILE_NAME"] == file, "FECHA"] = data.get("fecha", None)
    all_df.loc[all_df["FILE_NAME"] == file, "TOTAL"] = data.get("valor_total", None)

In [11]:
all_df.to_excel(main + "data/bills_2025_REVISADAS_v2.xlsx", index=False)